# Spindle — Scenario Packs

**Scenario Packs** are YAML-defined end-to-end data generation workflows. A single pack bundles:
- Domain + scale
- Simulation mode (`file_drop`, `stream`, or `hybrid`)
- Chaos configuration (optional)
- Validation gates
- Fabric target paths

Spindle ships **44 built-in packs** covering 11 industry verticals × 4 simulation types:

| Type | Description |
|------|-------------|
| `fd_daily_batch` | Daily file-drop with partitioning, manifest, done flag |
| `fd_schema_drift` | File-drop with chaos-injected schema drift |
| `st_realtime_events` | Pure streaming via EventEnvelope |
| `hy_stream_plus_microbatch` | Hybrid: batch files + stream events |

**Verticals:** retail · healthcare · financial · supply_chain · iot · hr · insurance · marketing · education · real_estate · manufacturing

In [1]:
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path

from sqllocks_spindle.packs.loader import PackLoader
from sqllocks_spindle.packs.runner import PackRunner
from sqllocks_spindle.packs.validator import PackValidator

loader = PackLoader()
_BUILTIN_PACKS_ROOT = loader._builtin_root

## 1. Browse built-in packs

In [3]:
packs = loader.list_builtin()
print(f'Total built-in packs: {len(packs)}')

pack_index = pd.DataFrame(packs).sort_values(['domain', 'pack_id'])
pack_index

Total built-in packs: 48


,domain,pack_id,path
0,education,fd_daily_batch,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
1,education,fd_schema_drift,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
2,education,hy_stream_plus_microbatch,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
3,education,st_realtime_events,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
4,financial,fd_daily_batch,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
5,financial,fd_schema_drift,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
6,financial,hy_stream_plus_microbatch,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
7,financial,st_realtime_events,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
8,healthcare,fd_daily_batch,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...
9,healthcare,fd_schema_drift,C:\Users\suref\OneDrive\VSCode\AzureClients\fo...


## 2. Inspect a pack

In [4]:
retail_pack_path = Path(_BUILTIN_PACKS_ROOT) / 'retail' / 'fd_daily_batch.yaml'
pack = PackLoader().load(retail_pack_path)

print(f'ID:          {pack.id}')
print(f'Kind:        {pack.kind}')
print(f'Domain:      {pack.domain}')
print(f'Description: {pack.description}')
print(f'Version:     {pack.pack_version}')

if pack.file_drop:
    print(f'\nFile-drop config:')
    print(f'  cadence:  {pack.file_drop.cadence}')
    print(f'  formats:  {pack.file_drop.formats}')

if pack.validation:
    print(f'\nValidation gates: {pack.validation.required_gates}')

ID:          fd_daily_batch
Kind:        file_drop
Domain:      retail
Description: Daily batch landing zone drop for Retail domain (bronze file landing).
Version:     1

File-drop config:
  cadence:  daily
  formats:  ['parquet', 'csv']

Validation gates: ['schema_conformance', 'referential_integrity']


## 3. Validate a pack against a domain

In [5]:
from sqllocks_spindle.domains.retail.retail import RetailDomain

vr = PackValidator().validate(pack, RetailDomain())
print(f'Valid:    {vr.is_valid}')
print(f'Errors:   {vr.errors}')
print(f'Warnings: {vr.warnings}')

Valid:    True
Errors:   []
Warnings: []


## 4. Run a file-drop pack

The `PackRunner` orchestrates the full workflow: generate data → simulate → validate → write output.

In [6]:
runner = PackRunner()

run_result = runner.run(
    pack=pack,
    domain=RetailDomain(),
    scale='fabric_demo',
    seed=42,
    base_path='/lakehouse/default/Files',
)

print(run_result.summary())
print(f'\nSuccess:       {run_result.is_success}')
print(f'Files written: {len(run_result.files_written)}')
print(f'Events:        {run_result.events_emitted:,}')
print(f'Elapsed:       {run_result.elapsed_time:.2f}s')

Pack Run: SUCCESS
  Pack:    fd_daily_batch
  Domain:  retail
  Scale:   fabric_demo
  Elapsed: 0.3s
  Files:   6
  Events:  0
  Validation gates:
    schema_conformance: PASS
    referential_integrity: PASS

Success:       True
Files written: 6
Events:        0
Elapsed:       0.26s


## 5. Run the schema-drift pack (chaos enabled)

In [7]:
drift_pack = PackLoader().load(Path(_BUILTIN_PACKS_ROOT) / 'retail' / 'fd_schema_drift.yaml')
print(f'Pack: {drift_pack.id}')
if drift_pack.failure_injection:
    print(f'Failure injection enabled: {drift_pack.failure_injection.enabled}')

drift_result = runner.run(
    pack=drift_pack,
    domain=RetailDomain(),
    scale='fabric_demo',
    seed=99,
    base_path='/lakehouse/default/Files',
)

print(f'\nSuccess:          {drift_result.is_success}')
print(f'Validation gates: {drift_result.validation_results}')
print(f'Errors:           {drift_result.errors}')

Pack: fd_schema_drift
Failure injection enabled: True

Success:          True
Validation gates: {'schema_conformance': True, 'referential_integrity': True}
Errors:           []


## 6. Run a streaming pack

In [8]:
stream_pack = PackLoader().load(Path(_BUILTIN_PACKS_ROOT) / 'retail' / 'st_realtime_events.yaml')

stream_result = runner.run(
    pack=stream_pack,
    domain=RetailDomain(),
    scale='fabric_demo',
    seed=42,
    base_path='/lakehouse/default/Files',
)

print(f'Success:        {stream_result.is_success}')
print(f'Events emitted: {stream_result.events_emitted:,}')

Success:        True

Events emitted: 1,170


## 7. Run a hybrid pack

In [9]:
hybrid_pack = PackLoader().load(Path(_BUILTIN_PACKS_ROOT) / 'retail' / 'hy_stream_plus_microbatch.yaml')

hybrid_result = runner.run(
    pack=hybrid_pack,
    domain=RetailDomain(),
    scale='fabric_demo',
    seed=42,
    base_path='/lakehouse/default/Files',
)

print(f'Success:        {hybrid_result.is_success}')
print(f'Files written:  {len(hybrid_result.files_written)}')
print(f'Events emitted: {hybrid_result.events_emitted:,}')

Success:        True
Files written:  8
Events emitted: 1,170


## 8. Run all packs for a domain (batch report)

A useful pattern for CI: loop all 4 pack types for a given domain, report which pass.

In [10]:
domain = RetailDomain()
domain_name = 'retail'
pack_types = ['fd_daily_batch', 'fd_schema_drift', 'st_realtime_events', 'hy_stream_plus_microbatch']

report_rows = []
for pack_type in pack_types:
    p = PackLoader().load(Path(_BUILTIN_PACKS_ROOT) / domain_name / f'{pack_type}.yaml')
    r = runner.run(pack=p, domain=domain, scale='fabric_demo', seed=42,
                   base_path='/lakehouse/default/Files')
    report_rows.append({
        'pack':    pack_type,
        'success': r.is_success,
        'files':   len(r.files_written),
        'events':  r.events_emitted,
        'elapsed': round(r.elapsed_time, 2),
        'errors':  r.errors,
    })

report_df = pd.DataFrame(report_rows)
report_df

,pack,success,files,events,elapsed,errors
0,fd_daily_batch,True,6,0,0.10,[]
1,fd_schema_drift,True,6,0,0.10,[]
2,st_realtime_events,True,1,1170,0.09,[]
3,hy_stream_plus_microbatch,True,8,1170,0.10,[]


## 9. Custom pack from YAML string

Write your own pack spec inline — useful for one-off testing scenarios.

In [11]:
import tempfile, textwrap

CUSTOM_PACK_YAML = textwrap.dedent("""\
    pack_version: 1
    id: my_custom_pack
    kind: file_drop
    domain: retail
    description: Custom daily batch for Notebook 08 demo

    fabric_targets:
      lakehouse_files_root: Files/landing/retail

    file_drop:
      cadence: daily
      partitioning: dt=YYYY-MM-DD
      formats: [parquet]
      entities: [customer, order]
      manifest:
        enabled: true
      done_flag:
        enabled: true
      lateness:
        enabled: false
      duplicates:
        enabled: false

    validation:
      required_gates: [schema_conformance]
""")

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(CUSTOM_PACK_YAML)
    tmp_path = f.name

custom_pack = PackLoader().load(tmp_path)
custom_result = runner.run(
    pack=custom_pack,
    domain=RetailDomain(),
    scale='fabric_demo',
    seed=42,
    base_path='/lakehouse/default/Files',
)

print(f'Pack ID:       {custom_pack.id}')
print(f'Success:       {custom_result.is_success}')
print(f'Files written: {len(custom_result.files_written)}')

Pack ID:       my_custom_pack
Success:       True
Files written: 3
